Download classification dataset:

https://archive.ics.uci.edu/dataset/53/iris


Data definition, from iris.names:

7. Attribute Information:
   1. sepal length in cm (sepal_length)
   2. sepal width in cm (sepal_width)
   3. petal length in cm (petal_length)
   4. petal width in cm (petal_width)
   5. class: ==> class
      -- Iris Setosa
      -- Iris Versicolour
      -- Iris Virginica


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split


In [3]:
column_names=["sepal_length", "sepal_width", "petal_length", "petal_width", "species"]
data = pd.read_csv('data/iris.data', names=column_names)

# make species a categorical column
data['species'] = data['species'].astype('category')

X = data.drop(["species"], axis=1) 
y = pd.get_dummies(data["species"])


Let's make sure the shapes are what we expected:

In [4]:
assert(X.shape == (150,4))
assert(y.shape == (150,3))

Let's look at the input data

In [5]:
X.info()
#y.value_counts

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sepal_length  150 non-null    float64
 1   sepal_width   150 non-null    float64
 2   petal_length  150 non-null    float64
 3   petal_width   150 non-null    float64
dtypes: float64(4)
memory usage: 4.8 KB


In [6]:
# y now has evenly-divided categories -- one of each type
y.value_counts()

Iris-setosa  Iris-versicolor  Iris-virginica
False        False            True              50
             True             False             50
True         False            False             50
Name: count, dtype: int64

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) 

Size of inputs
20% are reserved for testing


In [8]:
print("total number of examples   ", len(X))
print("number of training examples", len(X_train))
print("number of test examples    ", len(X_test))

total number of examples    150
number of training examples 120
number of test examples     30


Sample training data


In [9]:
print("** X_train info")
X_train.info()

print("\n** X_train values[0]")
print(X_train.values[0])

print("]\n** y info")
y_train.info()


** X_train info
<class 'pandas.core.frame.DataFrame'>
Index: 120 entries, 22 to 102
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sepal_length  120 non-null    float64
 1   sepal_width   120 non-null    float64
 2   petal_length  120 non-null    float64
 3   petal_width   120 non-null    float64
dtypes: float64(4)
memory usage: 4.7 KB

** X_train values[0]
[4.6 3.6 1.  0.2]
]
** y info
<class 'pandas.core.frame.DataFrame'>
Index: 120 entries, 22 to 102
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Iris-setosa      120 non-null    bool 
 1   Iris-versicolor  120 non-null    bool 
 2   Iris-virginica   120 non-null    bool 
dtypes: bool(3)
memory usage: 1.3 KB


In [10]:
from micrograd.engine import Value
from micrograd.nn import Neuron, Layer, MLP

In [11]:
model = MLP(4, [16, 16, 3]) # 2-layer neural network
print(model)
print("number of parameters", len(model.parameters()))

Multi-Layer Perceptron Structure:
 Layer 1/3 - Shape of the layer is: 4 X 16 (nin X nout)
	[Neuron 0: ReLUNeuron(4) -> w0 =-0.6368, w1 = 0.1542, w2 = 0.0226, w3 = 0.0354, b = 0.0000 ... Neuron 15: ReLUNeuron(4) -> w0 =-0.0484, w1 =-0.5315, w2 =-0.9828, w3 =-0.6192, b = 0.0000]
 Layer 2/3 - Shape of the layer is: 16 X 16 (nin X nout)
	[Neuron 0: ReLUNeuron(16) -> w0 =-0.0840, w1 =-0.2168, w2 =-0.7526, w3 = 0.8238, w4 =-0.3233, w5 = 0.7557, w6 =-0.8335, w7 = 0.6725, w8 =-0.5551, w9 = 0.5140, w10 = 0.2291, w11 = 0.1180, w12 = 0.6054, w13 = 0.4441, w14 = 0.8313, w15 =-0.1263, b = 0.0000 ... Neuron 15: ReLUNeuron(16) -> w0 =-0.7369, w1 =-0.0498, w2 = 0.9099, w3 =-0.5592, w4 = 0.0961, w5 = 0.2628, w6 = 0.1193, w7 = 0.1666, w8 =-0.1515, w9 =-0.6106, w10 =-0.0780, w11 =-0.8113, w12 = 0.9159, w13 =-0.3913, w14 = 0.5852, w15 = 0.4223, b = 0.0000]
 Layer 3/3 - Shape of the layer is: 16 X 3 (nin X nout)
	[Neuron 0: LinearNeuron(16) -> w0 = 0.4642, w1 =-0.2599, w2 =-0.3451, w3 = 0.3392, w4 =-0.3707

In [12]:
import random
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [13]:
def sparse_categorical_crossentropy(y_true, y_pred):
    """
    Calculates the sparse categorical cross-entropy loss.

    Args:
        y_true: Array of one-hot encoded true labels of shape 3xlen(y_pred).
        y_pred: Array of predicted probabilities (2D array).

    Returns:
        float: The calculated loss.
    """
    #assert y_true.shape == (120, 3)
    #assert y_pred.shape == (120, 3)
        
    #print("values[0] ", y_pred[0])
    #print(f'shape: {y_pred.shape}')
    #print(f'y_pred_values[0]: {y_pred[0]}\ny_true[0]:{y_true[0]}')
    # Calculate loss
    loss = -np.mean(np.sum(y_true * np.log(y_pred), axis=-1))
    return loss

In [14]:
import math 

def softmax(z):  
    """ Softmax converts a vector of values to a probability distribution.
    Args:
      z (ndarray (N,))  : input data, N features
    Returns:
      a (ndarray (N,))  : softmax of z
    """    
    ### START CODE HERE ### 
    l = np.array(z)
    e_pow_z = math.e**l
    e_pow_z_sum = np.sum(e_pow_z)
    a = e_pow_z / e_pow_z_sum
    


In [15]:
X_train_np = X_train.values
y_train_np = y_train.values

assert X_train_np.shape == (120, 4)
assert y_train_np.shape == (120, 3)

In [16]:
# loss function
def loss(batch_size=None):

    # inline DataLoader :)
    if batch_size is None:
        Xb, yb = X_train_np, y_train_np
    else:
        ri = np.random.permutation(X.shape[0])[:batch_size]
        Xb, yb = X_train_np[ri], y_train_np[ri]
    inputs = [list(map(Value, xrow)) for xrow in Xb]
    #print("Xb:", Xb[:1], "inputs:", inputs[:1])
    
    # forward the model to get scores
    scores = list(map(model, inputs))
    #print('** scores[0]', scores[0])
    #print("\n\n****** ", scores[0])
    #print("***************************************")
    def get_data(v):
        return [value.data for value in v]
    
    raw_scores = list(map(get_data, scores))
    #print('eaw_scores: ',raw_scores[0])
    #print("yb: ", yb, " scores: ", scores)
    data_loss = sparse_categorical_crossentropy(yb, raw_scores)
    #data_loss = sum(losses) * (1.0 / len(losses))
    # L2 regularization
    alpha = 1e-4
    reg_loss = alpha * sum((p*p for p in model.parameters()))
    total_loss = data_loss + reg_loss
    
    # also get accuracy
    accuracy = [yi ==  np.argmax(softmax(scores)) for yi, scores in zip(yb, raw_scores)]
    return total_loss, sum(accuracy) / len(accuracy)

total_loss, acc = loss()
print("initial total_loss: ", total_loss, " activation: ", acc)

initial total_loss:  Value(data=nan, grad=0)  activation:  [0.66666667 0.65833333 0.675     ]


/var/folders/_k/cq4d67l51tbcs72vqw3431xc0000gn/T/ipykernel_72119/1574376638.py:19: RuntimeWarning: invalid value encountered in log
  loss = -np.mean(np.sum(y_true * np.log(y_pred), axis=-1))


In [ ]:
# optimization
for k in range(100):
    
    # forward
    total_loss, acc = loss()
    
    # backward
    model.zero_grad()
    total_loss.backward()
    
    # update (sgd)
    learning_rate = 1.0 - 0.9*k/100    
    for p in model.parameters():
        p.data -= learning_rate * p.grad
    
    if k % 5 == 0:
        print(f"step {k} loss {total_loss.data}, accuracy {acc*100}%")


/var/folders/_k/cq4d67l51tbcs72vqw3431xc0000gn/T/ipykernel_72119/1574376638.py:19: RuntimeWarning: invalid value encountered in log
  loss = -np.mean(np.sum(y_true * np.log(y_pred), axis=-1))


step 0 loss nan, accuracy [66.66666667 65.83333333 67.5       ]%
step 5 loss nan, accuracy [66.66666667 65.83333333 67.5       ]%
step 10 loss nan, accuracy [66.66666667 65.83333333 67.5       ]%
step 15 loss nan, accuracy [66.66666667 65.83333333 67.5       ]%


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x104d4d850>>
Traceback (most recent call last):
  File "/Users/ovi/prog/machine-learning/learn-ai/micrograd/.venv/lib/python3.12/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


step 20 loss nan, accuracy [66.66666667 65.83333333 67.5       ]%
